In [1]:
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cpu --q

In [3]:
# Install torch if it's missing (needed for local environments)
import sys
!{sys.executable} -m pip install torch matplotlib numpy --q

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt

# --- 1. Dataset & Vocabulary ---
# A slightly larger dataset for better learning patterns
text_data = """
artificial intelligence is the future of technology
deep learning is a subset of machine learning
machine learning is a subset of artificial intelligence
neural networks are used in deep learning models
deep learning models mimic the human brain
technology is evolving through artificial intelligence
"""

# Preprocessing
words = text_data.lower().split()
vocab = sorted(list(set(words)))
word_to_ix = {word: i for i, word in enumerate(vocab)}
ix_to_word = {i: word for i, word in enumerate(vocab)}
vocab_size = len(vocab)

# Create Training Sequences: X (current_word) -> Y (next_word)
X_train = [word_to_ix[words[i]] for i in range(len(words) - 1)]
Y_train = [word_to_ix[words[i+1]] for i in range(len(words) - 1)]

# Convert to Tensors
X = torch.LongTensor(X_train).view(-1, 1) # Shape: [Batch, Seq_Len]
Y = torch.LongTensor(Y_train)

# --- 2. RNN Architecture ---
class SimpleRNN(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim):
        super(SimpleRNN, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        # batch_first=True makes input (batch, seq, feature)
        self.rnn = nn.RNN(embed_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, vocab_size)
        
    def forward(self, x, hidden=None):
        x = self.embedding(x)
        out, hidden = self.rnn(x, hidden)
        # We only need the output of the last time step
        out = self.fc(out[:, -1, :])
        return out, hidden

# --- 3. Training Logic ---
EMBED_DIM = 32
HIDDEN_DIM = 64
LR = 0.01
EPOCHS = 150

model = SimpleRNN(vocab_size, EMBED_DIM, HIDDEN_DIM)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LR)

print(f"Training on {vocab_size} unique words...")
loss_history = []

for epoch in range(EPOCHS):
    model.train()
    optimizer.zero_grad()
    
    # Forward pass
    output, _ = model(X)
    loss = criterion(output, Y)
    
    # Backward pass
    loss.backward()
    optimizer.step()
    
    loss_history.append(loss.item())
    if (epoch + 1) % 30 == 0:
        print(f"Epoch [{epoch+1}/{EPOCHS}] - Loss: {loss.item():.4f}")

# --- 4. Visualization ---
plt.figure(figsize=(10, 5))
plt.plot(loss_history, color='#2c3e50', linewidth=2)
plt.title('RNN Loss Convergence', fontsize=14, fontweight='bold')
plt.xlabel('Epochs')
plt.ylabel('Cross Entropy Loss')
plt.grid(True, alpha=0.3)
plt.show()

# --- 5. Prediction Function ---
def predict_next_word(current_word):
    model.eval()
    word = current_word.lower()
    if word not in word_to_ix:
        return "<Word not in Vocab>"
    
    with torch.no_grad():
        input_idx = torch.LongTensor([[word_to_ix[word]]])
        output, _ = model(input_idx)
        _, predicted_idx = torch.max(output, 1)
        return ix_to_word[predicted_idx.item()]

# --- Final Output ---
test_words = ["artificial", "deep", "machine", "future"]
print("\n" + "="*45)
print(f"{'Input Word':<20} | {'RNN Prediction'}")
print("-" * 45)
for w in test_words:
    print(f"{w:<20} | {predict_next_word(w)}")
print("="*45)